In [1]:
pip install selenium

Note: you may need to restart the kernel to use updated packages.


In [5]:
#Name: Nabiel Hakimi bin Hairulizam SN01082301
#Name: Danish Qayyum bin Muhamad Shahida SW01083631

import csv  # To save extracted data into CSV file
import time  # To control delays
from bs4 import BeautifulSoup  # To parse HTML content

from selenium import webdriver  # To automate browser
from selenium.webdriver.common.by import By  # To locate elements
from selenium.webdriver.chrome.options import Options  # Chrome settings
from selenium.webdriver.support.ui import WebDriverWait  # Wait for elements
from selenium.webdriver.support import expected_conditions as EC  # Conditions for waiting
from selenium.common.exceptions import (
    TimeoutException,
    NoSuchElementException,
    StaleElementReferenceException,
    NoSuchWindowException,
    WebDriverException
)


# Function to extract review details from one page
def scrape_page(soup, reviews):
    # Loop through all review blocks
    for review_block in soup.find_all('div', class_='item'):

        # Extract reviewer name
        name_el = review_block.find('span', class_='reviewer')
        name = name_el.get_text(strip=True) if name_el else ''

        # Extract review date
        date_el = review_block.find('span', class_='time')
        date = date_el.get_text(strip=True) if date_el else ''

        # Extract review content
        text_container = review_block.find(
            'div',
            class_='item-content-main-content-reviews-item'
        )
        text = text_container.get_text(" ", strip=True) if text_container else ''

        # Only save if review text exists
        if text:
            reviews.append({
                'Reviewer name': name,       # Store reviewer name
                'Review date': date,         # Store review date
                'Review content': text       # Store review text
            })


# Function to create and configure Chrome driver
def make_driver():
    opts = Options()

    # Uncomment below if you want headless mode (no browser UI)
    # opts.add_argument("--headless=new")

    opts.add_argument("--start-maximized")  # Start browser maximized
    opts.add_argument("--disable-gpu")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")

    # Reduce automation detection
    opts.add_experimental_option("excludeSwitches", ["enable-automation"])
    opts.add_experimental_option("useAutomationExtension", False)
    opts.add_argument("--disable-blink-features=AutomationControlled")

    # Set custom user agent
    opts.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/145.0.0.0 Safari/537.36"
    )

    driver = webdriver.Chrome(options=opts)

    # Hide webdriver flag using JavaScript
    driver.execute_cdp_cmd(
        "Page.addScriptToEvaluateOnNewDocument",
        {"source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"}
    )

    return driver


# Ensure driver switches to active/latest tab
def ensure_active_window(driver):
    try:
        handles = driver.window_handles
        if not handles:
            return
        driver.switch_to.window(handles[-1])
    except Exception:
        pass


# Scroll page to trigger lazy loading of reviews
def scroll_reviews_into_view(driver):
    driver.execute_script(
        "window.scrollTo(0, document.body.scrollHeight * 0.6);"
    )
    time.sleep(2)


# Main function to scrape multiple pages
def scrape_all_pages(url, max_pages=6):
    reviews = []  # List to store all reviews
    driver = make_driver()
    wait = WebDriverWait(driver, 25)  # Wait up to 25 seconds

    try:
        driver.get(url)  # Open product page
        time.sleep(3)
        ensure_active_window(driver)

        current_page = 1

        # Loop through review pages
        while current_page <= max_pages:
            ensure_active_window(driver)
            scroll_reviews_into_view(driver)

            # Wait until review items appear
            try:
                wait.until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, "div.item"))
                )
            except TimeoutException:
                # Extra scroll if not loaded
                driver.execute_script(
                    "window.scrollTo(0, document.body.scrollHeight * 0.9);"
                )
                time.sleep(2)
                wait.until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, "div.item"))
                )

            # Parse page source with BeautifulSoup
            soup = BeautifulSoup(driver.page_source, "html.parser")
            scrape_page(soup, reviews)

            print(f"Scraped page {current_page}, total reviews: {len(reviews)}")

            # Stop if reached max page
            if current_page == max_pages:
                break

            # Locate next page button
            try:
                next_btn = wait.until(
                    EC.presence_of_element_located(
                        (By.CSS_SELECTOR, ".iweb-pagination-next")
                    )
                )
            except TimeoutException:
                print("No next button found. Stopping.")
                break

            # Stop if next button disabled
            cls = next_btn.get_attribute("class") or ""
            if "disabled" in cls:
                print("Next button disabled. Stopping.")
                break

            # Get first item to detect page refresh
            first_item = driver.find_element(By.CSS_SELECTOR, "div.item")

            try:
                wait.until(
                    EC.element_to_be_clickable(
                        (By.CSS_SELECTOR, ".iweb-pagination-next")
                    )
                )

                # Click next page
                try:
                    next_btn.click()
                except Exception:
                    driver.execute_script("arguments[0].click();", next_btn)

                # Wait until old content disappears
                wait.until(EC.staleness_of(first_item))
                time.sleep(2)
                current_page += 1

            except (StaleElementReferenceException, TimeoutException):
                current_page += 1
                time.sleep(2)

    except NoSuchWindowException:
        print("Browser window closed/crashed during scraping.")
    except WebDriverException as e:
        print("WebDriver error:", e)
    finally:
        # Close browser safely
        try:
            driver.quit()
        except Exception:
            pass

    return reviews


# Product URL
base_url = "https://www.lazada.com.my/products/pdp-i3796853738-s21945790347.html?c=&channelLpJumpArgs=&clickTrackInfo=query%253Acable%252Btypec%252Bapple%253Bnid%253A3796853738%253Bsrc%253ALazadaMainSrp%253Brn%253A6f79c0a30e13b0fa34926ca0f2c19e5b%253Bregion%253Amy%253Bsku%253A3796853738_MY%253Bprice%253A129%253Bclient%253Adesktop%253Bsupplier_id%253A100149888%253Bsession_id%253A%253Bbiz_source%253Ah5_hp%253Bslot%253A0%253Butlog_bucket_id%253A470687%253Basc_category_id%253A11162%253Bitem_id%253A3796853738%253Bsku_id%253A21945790347%253Bshop_id%253A153210%253BtemplateInfo%253A107880_E%2523-1_A3_C%2523&freeshipping=1&fs_ab=2&fuse_fs=&lang=en&location=Selangor&price=129&priceCompare=skuId%3A21945790347%3Bsource%3Alazada-search-voucher%3Bsn%3A6f79c0a30e13b0fa34926ca0f2c19e5b%3BoriginPrice%3A12900%3BdisplayPrice%3A12900%3BsinglePromotionId%3A900000104726102%3BsingleToolCode%3ApromPrice%3BvoucherPricePlugin%3A0%3Btimestamp%3A1772458268840&ratingscore=5.0&request_id=6f79c0a30e13b0fa34926ca0f2c19e5b&review=139&sale=487&search=1&source=search&spm=a2o4k.searchlist.list.0&stock=1"

# Scrape up to 5 pages
reviews = scrape_all_pages(base_url, max_pages=5)

# Save extracted data into CSV file
with open("review-lazada-extracted.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["Reviewer name", "Review date", "Review content"])
    writer.writeheader()  # Write column headers
    writer.writerows(reviews)  # Write all reviews

print("Done! Saved:", len(reviews), "reviews")

Scraped page 1, total reviews: 5
Scraped page 2, total reviews: 10
Scraped page 3, total reviews: 15
Scraped page 4, total reviews: 20
Scraped page 5, total reviews: 25
Done! Saved: 25 reviews
